# Orfipy ORF Prediction

![Orfipy ORF Prediction](https://proto-bio.github.io/proto-assets/images/tool/orfipy/hero.png)

This notebook demonstrates `run_orfipy_prediction`, which identifies open reading frames in DNA sequences by scanning for start and stop codons across all reading frames. It covers single-sequence prediction, custom codon and strand configuration, batch processing, and result inspection.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("orfipy")
display_overview("orfipy")
display_docs_section("orfipy", "Background")

## Available tools

In [ ]:
display_available_tools("orfipy")

### `run_orfipy_prediction`

`run_orfipy_prediction` scans DNA sequences for open reading frames using Orfipy, a fast Cython-based ORF finder. For each input sequence it returns all ORFs that pass the configured start codon, stop codon, length, and strand filters. Each predicted ORF carries the amino acid translation, nucleotide sequence, 1-indexed coordinates, strand, reading frame, and GC content. Batch inputs are processed together and results are returned as a nested list keyed by input sequence, so downstream code can group or filter ORFs by their parent sequence.

In [ ]:
from proto_tools import OrfipyInput, OrfipyConfig, OrfipyOutput, run_orfipy_prediction

In [ ]:
# Display input docs
display_api_reference("orfipy", "input", "run_orfipy_prediction")

# A short DNA sequence with a known ORF (hemoglobin alpha fragment)
inputs = OrfipyInput(sequences="ATGGTGCTGAGCCCGGCGGACAAGACCAACGTGAAGGCGGCGTGGGGCAAGTGA")

In [ ]:
# Display config docs
display_api_reference("orfipy", "config", "run_orfipy_prediction")

# Forward strand only, ATG start codon only, minimum 30 nucleotides | see docs above for all fields
config = OrfipyConfig(
    start_codons=["ATG"],
    min_len=30,
    strand="f",
    include_stop=True,
)

In [ ]:
# Run the prediction
result = run_orfipy_prediction(inputs, config)

In [ ]:
# Display output docs
display_api_reference("orfipy", "output", "run_orfipy_prediction")

# Inspect predicted ORFs from the single input sequence
print(f"Found {result.num_orfs} ORFs")
for orf_list in result.predicted_orfs:
    for orf in orf_list:
        print(f"  {orf.id}: {orf.amino_acid_sequence} ({orf.amino_acid_length} aa, {orf.strand} strand, frame {orf.frame})")

# Demonstrate batch processing over multiple sequences
sequences = [
    "ATGGTGCTGAGCCCGGCGGACAAGACCAACGTGAAGGCGGCGTGGGGCAAGTGA",  # human hemoglobin alpha CDS
    "ATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTGACTGGGAAAACCCTGGCGTTACCCAACTTAATCGCCTTGCAGCACATCCCCCTTTCGCCAGCTGGCGTAATAGCGAAGAGGCCCGCACCGATCGCCCTTCCCAACAGTTGCGCAGCCTGAATGGCGAATGGCGCTTTGCCTGGTTTCCGGCACCAGAAGCGGTGCCG",  # E. coli lacZ N-terminal coding fragment
    "ATGAGTAAAGGAGAAGAACTTTTCACTGGAGTTGTCCCAATTCTTGTTGAATTAGATGGTGATGTTAATGGGCACAAATTTTCTGTCAGTGGAGAGGGTGAAGGTGATGCAACATACGGAAAACTTACCCTTAAATTTATTTGCACTACTGGAAAACTACCTGTTCCATGG",  # GFP N-terminal coding fragment
]
batch_inputs = OrfipyInput(
    sequences=sequences,
)
batch_config = OrfipyConfig(min_len=12)
batch_result = run_orfipy_prediction(batch_inputs, batch_config)

print(f"\nBatch: {batch_result.num_orfs} total ORFs across {len(sequences)} sequences")
print(f"ORFs per sequence: {batch_result.num_orfs_per_sequence}")

# Inspect an individual ORF
orf = batch_result.predicted_orfs[0][0]
print(f"\nFirst ORF details:")
print(f"  Parent ID:   {orf.parent_id}")
print(f"  Position:    {orf.nucleotide_start}-{orf.nucleotide_end}")
print(f"  AA length:   {orf.amino_acid_length}")
print(f"  GC content:  {orf.gc_content:.1f}%")
print(f"  Protein:     {orf.amino_acid_sequence}")

## Export Results

Predicted ORFs can be exported to CSV or JSON for tabular analysis, or to FASTA format (FAA for protein sequences, FNA for nucleotide sequences) for use in downstream bioinformatics pipelines.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

batch_result.export("orfipy_results", export_path=output_dir, file_format="csv")
print(f"Exported CSV to {output_dir / 'orfipy_results.csv'}")

batch_result.export("orfipy_results", export_path=output_dir, file_format="faa")
print(f"Exported protein FASTA to {output_dir / 'orfipy_results.faa'}")